# Performance analysis (issue #40)

Exploratory analysis of the 85-record dataset (#38): CreativeIR structural
features vs public engagement. Offline only — no model API calls.

Caveats: n=85, single creator, observational data, many features tested
(multiple-comparison risk). Correlations are NOT causal.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "scripts"))
from performance import TARGETS, fit_baseline, spearman_report

df = pd.read_parquet(repo_root / "dataset" / "features.parquet")
print(f"records: {len(df)}")
print(df[["duration_seconds", "shot_count", "log1p_views", "log1p_like_rate"]].describe().round(2).to_string())

In [ ]:
feature_cols = [
    c for c in df.columns
    if c not in {"video_id", "views", *TARGETS} and df[c].notna().sum() >= 10 and df[c].nunique() >= 2
]
report = spearman_report(df, feature_cols, TARGETS)
print(report.head(25).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

top = report[report["target"] == "like_rate"].dropna().head(10)
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(top["feature"][::-1], top["spearman"][::-1])
ax.set_title("Spearman correlation with like_rate (top 10)")
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
for target in ("log1p_views", "log1p_like_rate"):
    result = fit_baseline(df, feature_cols, target=target)
    print(f"=== {target} ===")
    print(f"n={result['n']}  CV R2={result['cv_r2']:.3f}  (median baseline R2=0)")
    for name, coef in result["top_coefficients"][:8]:
        print(f"  {coef:+.4f}  {name}")
    print()

## Reading

- **Views are NOT predicted by structure alone** (CV R2 <= 0 on log1p_views):
  reach is dominated by distribution factors (series following, topicality,
  external traffic) absent from CreativeIR.
- **Like rate IS strongly structural** (CV R2 ~0.43): creator dialogue (+),
  social-proof mechanics (+), sound-effects-heavy mixes (-), slower pacing
  (+ like rate), challenge-format roles (+).
- This is exactly what the future generator needs: engagement-rate-relevant
  creative levers, not view-count fortune telling.